# 20. 최종 모델 학습·테스트 (M_maha_free) — 팀 공유용

**모델**: unlabeled 정상 생산 샷만으로 학습한 비지도 이상탐지 (라벨은 평가에만 사용)

| 단계 | 내용 |
|---|---|
| 학습 데이터 | unlabeled 1차, 2020-09-01~11-30, 우진2호기 / labeled 샷·Run 시간창(±60분)·이상 기록·10샷 미만 Run·0 모드(배럴1 < 100℃) 제거 |
| 입력 변수 | 인과적 파생변수 17개 — 같은 Run 직전 20샷 대비 편차(dev_*), 충전 이상, 열 상태(상승량·기울기), 흐름저항 편차 |
| 학습 | reference 중앙값·IQR 스케일 → ±20 클리핑 → Ledoit-Wolf 공분산 |
| 점수 | Mahalanobis 거리 → reference 대비 백분위(pct) |
| 임계값 | pct ≥ q (q = 1 − 정상 오경보율, 라벨 미사용) |

**필요 파일** (notebooks 폴더 기준): `linked/linked_unlabeled_{cn7,rg3}.csv`, `linked/linked_labeled.csv`, `base/{cn7,rg3}_shots.csv`, `base/reference_scales.json`
**패키지**: numpy, pandas, scikit-learn — `final_model.py` 하나만 있으면 됨

**출력** (`./outputs/final_model/`): 지표 CSV, 샷별 점수 `{p}_scores.csv`, 학습된 모델 `{p}_final_model.pkl`

In [1]:
import pandas as pd
from final_model import run_all, compare_with_cache, FEATURES
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)

R = run_all()          # 학습 + 테스트 (수 분 소요: Run별 편차·기울기 계산)

[cn7] reference 5,179샷 학습 → labeled 1,979샷 점수 완료
[rg3] reference 4,359샷 학습 → labeled 629샷 점수 완료


## 0. 재현 확인 (18번 수치와 같은지) + 데이터 처리 기록

In [2]:
display(R['check'])
display(R['data_log'])

,제품,틀,AP(이번),AP(18번),일치
0,cn7,①,0.373,0.373,O
1,cn7,②,0.212,0.212,O
2,rg3,①,0.024,0.024,O
3,rg3,②,0.064,0.064,O


,unlabeled 샷,labeled 시간대 제거·기간 적용 후,이상 기록 제거,0 모드 제거,reference 샷,reference Run,제품,기준값 출처
0,28744,18068,1634,11174,5179,4,cn7,reference_scales.json
1,29633,4441,0,47,4359,6,rg3,reference_scales.json


## 1. 평가 ① 공식 2차 · 행 단위 (팀원 평가 틀)
- **AP**: 주 지표 / **무작위 평균 대비**·**유의**: 무작위 순위 1,000번의 97.5%를 넘으면 O
- **상위 k%**: 점수 상위 k% 행만 검사할 때 잡히는 불량 행
- **F1@5%·10%**: 상위 k%를 불량 판정 (정직한 F1) / **F1(최적,참고)**: 정답 보고 고른 임계값 → 참고용
- **상한 F1**: LH/RH 같은 입력 행은 같은 판정 → 어떤 모델도 못 넘는 값

In [3]:
display(R['frame1'].round(3))

,제품,행,불량 행,AP,무작위 평균 대비,무작위 상단(97.5%),유의,상위1%,상위2%,상위5%,상위10%,F1@5%,TP@5%,F1@10%,TP@10%,"F1(최적,참고)",상한 F1,F1@5% / 상한,F1(최적) / 상한
0,cn7,1211,17,0.373,19.845,0.041,O,6/17,6/17,6/17,8/17,0.154,6,0.116,8,0.522,0.756,0.204,0.691
1,rg3,1182,25,0.024,0.892,0.054,-,0/25,0/25,1/25,2/25,0.024,1,0.028,2,0.056,0.667,0.036,0.083


## 2. 평가 ② labeled 전체 · 샷 단위 · 초기허용불량 제외
- **AP / AP_lo~AP_hi**: AP와 부트스트랩 95% 구간, **AP/불량률** = 무작위 대비 배수
- **Run별 상위 k%**: Run마다 상위 k% 검사 / **전체 상위 k%**: 파일 전체 상위 k% 검사
- **AP_가스 / AP_미성형**: 유형별 성능

In [4]:
display(R['frame2'].round(3))

,제품,샷,불량,불량률,AP,AP_lo,AP_hi,AP/불량률,Run별 상위5%,Run별 상위10%,전체 상위5%,F1@5%,전체 상위10%,F1@10%,AP_가스,Run별 상위10%_가스,AP_미성형,Run별 상위10%_미성형
0,cn7,1974,15,0.008,0.212,0.011,0.416,27.843,3/15,4/15,4/15,0.070,5/15,0.047,0.007,1/11,0.751,3/4
1,rg3,627,32,0.051,0.064,0.051,0.101,1.247,1/32,3/32,3/32,0.063,3/32,0.063,0.050,2/22,0.021,1/10


## 3. 라벨 없이 정한 임계값의 경보 성능 (② 기준)
q = 0.999 → 정상 샷의 0.1%만 경보. **경보 비율 ≈ 1 − q** 이면 임계값 보정이 잘 된 것

In [5]:
display(R['threshold'].round(3))

,제품,q,정상 오경보율(가정),경보 수,경보 비율,TP,FP,FN,TN,Precision,Recall,F1,가스 탐지,미성형계열 탐지
0,cn7,0.950,5%,105,0.053,4,101,11,1858,0.038,0.267,0.067,1,3
1,cn7,0.980,2%,45,0.023,3,42,12,1917,0.067,0.200,0.100,0,3
2,cn7,0.990,1%,29,0.015,3,26,12,1933,0.103,0.200,0.136,0,3
3,cn7,0.995,0.5%,19,0.010,3,16,12,1943,0.158,0.200,0.176,0,3
4,cn7,0.999,0.1%,8,0.004,3,5,12,1954,0.375,0.200,0.261,0,3
5,rg3,0.950,5%,45,0.072,3,42,29,553,0.067,0.094,0.078,3,0
6,rg3,0.980,2%,20,0.032,1,19,31,576,0.050,0.031,0.038,1,0
7,rg3,0.990,1%,9,0.014,0,9,32,586,0.000,0.000,0.000,0,0
8,rg3,0.995,0.5%,6,0.010,0,6,32,589,0.000,0.000,0.000,0,0
9,rg3,0.999,0.1%,3,0.005,0,3,32,592,0.000,0.000,0.000,0,0


## PR-AUC · PR 곡선
| 지표 | 계산 | 특징 |
|---|---|---|
| **AP** (= 계단식 PR-AUC) | 불량을 하나씩 만날 때의 정밀도 평균 | sklearn·논문 표준 → 주 지표 |
| **PR-AUC(사다리꼴)** | PR 곡선 점 사이를 직선으로 이어 면적 계산 | 불량이 적으면 낙관적으로 나올 수 있음 → 참고 |

무작위 모델의 PR-AUC = 불량률 (그래프 회색 점선). 빨간 점 = 라벨 없이 정한 임계값(q=0.99, 0.999)의 실제 운영점

In [ ]:
from final_model import plot_pr
c1 = ['제품', '행', '불량 행', 'AP', 'PR-AUC(사다리꼴)', '무작위 PR-AUC(=불량률)', '무작위 평균 대비', '유의']
c2 = ['제품', '샷', '불량', '불량률', 'AP', 'AP_lo', 'AP_hi', 'PR-AUC(사다리꼴)', 'AP/불량률']
print('① 공식 2차 · 행'); display(R['frame1'][c1].round(3))
print('② 전체 · 샷 · 초기허용 제외'); display(R['frame2'][c2].round(3))
plot_pr(R, save_dir='./outputs/final_model')

## 4. (선택) 기존 파생변수 캐시와 비교 — `base/{p}_features.csv`가 있을 때만

In [6]:
for p in ['cn7', 'rg3']:
    print(p); display(compare_with_cache(R, p))

cn7


,변수,최대 절대차,NaN 불일치
0,dev_Max_Injection_Speed,4.440892e-16,0
1,dev_Filling_Time,3.552714e-15,0
2,dev_Max_Injection_Pressure,7.105427e-15,0
3,dev_Max_Switch_Over_Pressure,1.776357e-15,0
4,dev_Average_Back_Pressure,3.552714e-15,0
5,dev_Cushion_Position,0.000000e+00,0
6,dev_Plasticizing_Time,3.552714e-15,0
7,dev_Cycle_Time,2.842171e-14,0
8,fill_anomaly,7.105427e-15,0
9,ev_max_abs_dev,2.842171e-14,0


rg3


,변수,최대 절대차,NaN 불일치
0,dev_Max_Injection_Speed,4.440892e-16,0
1,dev_Filling_Time,0.000000e+00,0
2,dev_Max_Injection_Pressure,4.440892e-16,0
3,dev_Max_Switch_Over_Pressure,3.552714e-15,0
4,dev_Average_Back_Pressure,4.440892e-16,0
5,dev_Cushion_Position,5.551115e-17,0
6,dev_Plasticizing_Time,4.440892e-16,0
7,dev_Cycle_Time,8.881784e-16,0
8,fill_anomaly,2.220446e-16,0
9,ev_max_abs_dev,3.552714e-15,0


## 5. (참고) 저장된 모델로 새 샷 점수 매기기
```python
from final_model import score_new_shots
f = score_new_shots(new_shots, 'cn7')      # new_shots: run_id, TimeStamp, *_raw (같은 Run 과거 샷 포함)
alarm = f[f['pct'] >= 0.999]               # 검사 대상
```
⚠ 비가동(0 모드, 배럴1 < 100℃) 샷은 점수 계산 전에 걸러낼 것